In [1]:
import pandas as pd

In [2]:
titanic_df=pd.read_csv("titanic.csv")

In [3]:
titanic_df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [4]:
titanic_df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
titanic_df=titanic_df[["Survived","Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"]]

In [6]:
titanic_df.head(2)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C


In [7]:
from sklearn.model_selection import train_test_split

x=titanic_df.drop(columns=["Survived"])
y=titanic_df[["Survived"]]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [17]:
x_train.head(3)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.500,S
733,2,male,23.0,0,0,13.000,S
382,3,male,32.0,0,0,7.925,S


In [19]:
x_train.isnull().sum()
#age and embarked having missing value

Pclass        0
Sex           0
Age         140
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


In [22]:
#pipeline vai single transformer,-->all transformer execute at same time

In [32]:
trf=ColumnTransformer([("imputer_age",SimpleImputer(),[2]),
                       ("imputer_embarked",SimpleImputer(strategy="most_frequent"),[6]),
                       ("ohe_sex_embarked",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),[1,6])
                       ],remainder="passthrough") 
#now the age column having missing value will filled with the means value ,'cause of SimpleImputer Transformer using parameter,by-default have--> strategy="mean".

In [35]:
pipe=Pipeline([("trf",trf)])

In [36]:
pipe.fit_transform(x_train)

array([[45.5, 'S', 0.0, ..., 0.0, 0.0, 28.5],
       [23.0, 'S', 0.0, ..., 0.0, 0.0, 13.0],
       [32.0, 'S', 0.0, ..., 0.0, 0.0, 7.925],
       ...,
       [41.0, 'S', 0.0, ..., 2.0, 0.0, 14.1083],
       [14.0, 'S', 1.0, ..., 1.0, 2.0, 120.0],
       [21.0, 'S', 0.0, ..., 0.0, 1.0, 77.2875]],
      shape=(712, 12), dtype=object)

In [38]:
pipe.get_feature_names_out()

array(['imputer_age__Age', 'imputer_embarked__Embarked',
       'ohe_sex_embarked__Sex_female', 'ohe_sex_embarked__Sex_male',
       'ohe_sex_embarked__Embarked_C', 'ohe_sex_embarked__Embarked_Q',
       'ohe_sex_embarked__Embarked_S', 'ohe_sex_embarked__Embarked_nan',
       'remainder__Pclass', 'remainder__SibSp', 'remainder__Parch',
       'remainder__Fare'], dtype=object)

In [ ]:
#here,all transformer execute same time,so the ohe_embarked used original data.the original embarked having missing value so i=ohe used that embarked transformer.

In [39]:
#ALWAYS REMEMBER:while doing column transformer u do transformer ron column that column come first then after that remainder column comes

In [10]:
x_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [21]:
embarked_pipe=Pipeline([("imputer_embarked",SimpleImputer(strategy="most_frequent")),
                        ("ohe_embarked",OneHotEncoder(sparse_output=False,handle_unknown="ignore"))])

In [22]:
trf=ColumnTransformer([("imputer_age",SimpleImputer(),[2]),
                       ("ohe_sex",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),[1]),
                       ("embarked_imp_ohe",embarked_pipe,[6])],remainder="passthrough")

In [23]:
pip=Pipeline([("trf",trf)])

In [26]:
pip_df=pip.fit_transform(x_train)

In [27]:
pip_df

array([[ 45.5   ,   0.    ,   1.    , ...,   0.    ,   0.    ,  28.5   ],
       [ 23.    ,   0.    ,   1.    , ...,   0.    ,   0.    ,  13.    ],
       [ 32.    ,   0.    ,   1.    , ...,   0.    ,   0.    ,   7.925 ],
       ...,
       [ 41.    ,   0.    ,   1.    , ...,   2.    ,   0.    ,  14.1083],
       [ 14.    ,   1.    ,   0.    , ...,   1.    ,   2.    , 120.    ],
       [ 21.    ,   0.    ,   1.    , ...,   0.    ,   1.    ,  77.2875]],
      shape=(712, 10))

In [25]:
trf.get_feature_names_out()

array(['imputer_age__Age', 'ohe_sex__Sex_female', 'ohe_sex__Sex_male',
       'embarked_imp_ohe__Embarked_C', 'embarked_imp_ohe__Embarked_Q',
       'embarked_imp_ohe__Embarked_S', 'remainder__Pclass',
       'remainder__SibSp', 'remainder__Parch', 'remainder__Fare'],
      dtype=object)

In [33]:
x_train.index

Index([331, 733, 382, 704, 813, 118, 536, 361,  29,  55,
       ...
       121, 614,  20, 700,  71, 106, 270, 860, 435, 102],
      dtype='int64', length=712)

In [30]:
pd.DataFrame(pip_df,columns=trf.get_feature_names_out(),index=x_train.index)

,imputer_age__Age,ohe_sex__Sex_female,ohe_sex__Sex_male,embarked_imp_ohe__Embarked_C,embarked_imp_ohe__Embarked_Q,embarked_imp_ohe__Embarked_S,remainder__Pclass,remainder__SibSp,remainder__Parch,remainder__Fare
331,45.500000,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,28.5000
733,23.000000,0.0,1.0,0.0,0.0,1.0,2.0,0.0,0.0,13.0000
382,32.000000,0.0,1.0,0.0,0.0,1.0,3.0,0.0,0.0,7.9250
704,26.000000,0.0,1.0,0.0,0.0,1.0,3.0,1.0,0.0,7.8542
813,6.000000,1.0,0.0,0.0,0.0,1.0,3.0,4.0,2.0,31.2750
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,1.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,7.6500
270,29.498846,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,31.0000
860,41.000000,0.0,1.0,0.0,0.0,1.0,3.0,2.0,0.0,14.1083
435,14.000000,1.0,0.0,0.0,0.0,1.0,1.0,1.0,2.0,120.0000
